# Programmatic Access to Interface Data with the PDBe PISA API

This notebook reproduces the **Programmatic Access to Interface Data with PDBe PISA API**
section of *"PDBe PISA: A Protocol for Enhanced Analysis of Macromolecular Interactions"*
(Díaz Leines et al., 2026).

**PISA** (Proteins, Interfaces, Structures and Assemblies; Krissinel & Henrick, 2007) is a
standard tool for identifying intermolecular interfaces and predicting biologically relevant
macromolecular assemblies from 3D structural data. For every interface in a crystal lattice it
reports a range of physicochemical properties — interface area, solvation free-energy gain,
hydrogen bonds, salt bridges, and the free energy of dissociation (ΔG<sup>diss</sup>) — together
with a *p*-value that statistically assesses interface specificity (Krissinel, 2010). This
protocol uses the updated PISA implementation distributed with CCP4 v9.0 and used by PDBe
([PDBe-KB/pisa](https://github.com/PDBe-KB/pisa)), which provides expanded interface annotations
and a broadened catalogue of atom–atom interaction types. The PDBe PISA REST API exposes these
results programmatically, so they can be queried, tabulated, and compared at scale.

The worked example throughout is the **yeast 20S proteasome (PDB `1iru`)** — a 28-subunit
assembly with D7 symmetry. All requests are issued against the live PDBe PISA service at
`https://www.ebi.ac.uk/pdbe/api/pisa/...`; an active internet connection is therefore required,
and the figures reflect the current weekly PISA release rather than a pinned version.

**References**
- Díaz Leines et al. (2026) *PDBe PISA: A Protocol for Enhanced Analysis of Macromolecular Interactions* — the source protocol reproduced here.
- Krissinel, E. & Henrick, K. (2007) *J. Mol. Biol.* **372**, 774–797 — the PISA method.
- Krissinel, E. (2010) *J. Comput. Chem.* **31**, 133–143 — complexation significance / *p*-value.
- PDBe PISA REST API documentation (Apiary): https://pdbeapi.docs.apiary.io/
- Reference Colab notebook: the `pisa/v1.0.0` tag of the
  [PDBeurope/pdbe-notebooks](https://github.com/PDBeurope/pdbe-notebooks) repository

## Section 0 — Setup

Install the third-party dependencies. `requests` issues the HTTP calls to the PDBe REST API,
`pandas` structures the results into tables, `tabulate` supports formatted text output, and
`IPython.display` (bundled with Jupyter) renders DataFrames inline.

In [13]:
!pip install requests pandas tabulate


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Section 1 — Data for a single assembly

The PISA *assembly* endpoint returns the summary characterisation of one macromolecular assembly
of a PDB entry: its composition, total interface count, surface areas, and the thermodynamic
estimates that quantify how strongly the assembly is held together. We begin with assembly `1`
of the proteasome entry `1iru`.

### Cell 1.1 — Retrieve a single assembly (`1iru`, assembly 1)

The request URL is composed from the API base, the PDB identifier, and the assembly number, and
issued as an HTTP GET. On a successful response (HTTP 200) the complete JSON payload is
serialised with indentation, so that the structure and field names PISA returns can be
inspected directly.

In [14]:
import requests, sys, json

API_point = 'https://www.ebi.ac.uk/pdbe/api/pisa/assembly/'
query_pdbid = '1iru'
query_assembly_id = '1'

response = requests.get(f'{API_point}{query_pdbid}/{query_assembly_id}')

if response.status_code == 200:
    print(json.dumps(response.json(), indent=3))

{
   "1iru": {
      "assembly_id": "1",
      "pisa_version": "2.0",
      "assembly": {
         "id": "1",
         "size": "56",
         "interface_count": 82,
         "score": "",
         "macromolecular_size": "28",
         "dissociation_energy": 69.0,
         "accessible_surface_area": 212622.4,
         "buried_surface_area": 113317.18,
         "entropy": 28.55,
         "dissociation_area": 7116.05,
         "solvation_energy_gain": -637.29,
         "number_of_uc": "0",
         "number_of_dissociated_elements": "3",
         "symmetry_number": "2",
         "formula": "\n        A(2)B(2)C(2)D(2)E(2)F(2)G(2)H(2)I(2)J(2)K(2)L(2)M(2)N(2)a(28)\n      ",
         "composition": "AOBPCQDRESFTGUHVIWJXKYLZ1M2N[MG](28)",
         "R350": ""
      }
   }
}


### Cell 1.2 — Summarise the assembly metrics

From the response we read the `assembly` record and report the quantities that describe the
assembly as a whole, following the parameter definitions given in the paper:

- **Components / macromolecular size** — `macromolecular_size` is the number of macromolecular
  chains (e.g. protein, nucleic acid) in the assembly.
- **Interface count** — the total number of unique pairwise interfaces constituting the assembly.
- **Accessible surface area (Å²)** — the total solvent-accessible surface area of the assembly.
- **Buried surface area, BSA (Å²)** — the total solvent-accessible surface area buried upon
  formation of all interfaces in the assembly; it quantifies the overall extent of intermolecular
  contact.
- **Dissociation free energy, ΔG<sup>diss</sup> (kcal/mol)** — the free energy of complex
  dissociation, i.e. the difference between the associated and dissociated states, and the primary
  indicator of overall assembly stability. A positive value means dissociation is non-spontaneous
  (an external driving force would be required to dissociate the assembly); larger positive values
  denote greater stability.
- **Entropy (kcal/mol)** — the rigid-body entropy change upon dissociation, corresponding to the
  lowest-free-energy route to separating the complex into stable sub-complexes or monomers.

The values are printed as a concise assembly summary.

In [15]:
response_data = response.json()
pdb_id = list(response_data.keys())[0]
assembly_data = response_data[pdb_id]['assembly']

total_size = assembly_data.get('size', 'N/A')
macromolecular_size = assembly_data.get('macromolecular_size', 'N/A')
buried_surface_area = assembly_data.get('buried_surface_area', 'N/A')
accessible_surface_area = assembly_data.get('accessible_surface_area', 'N/A')
no_interfaces = assembly_data.get('interface_count', 'N/A')
dissociation_energy = assembly_data.get('dissociation_energy', 'N/A')
entropy = assembly_data.get('entropy', 'N/A')

print(f"--- Assembly Summary for {pdb_id} (Assembly {query_assembly_id}) ---")
print(f"  Components: {total_size} total ({macromolecular_size}-mer protein chains)")
print(f"  Interfaces: {no_interfaces} total")
print(f"  Accessible Surface Area: {accessible_surface_area} Å²")
print(f"  Buried Surface Area: {buried_surface_area} Å²")
print(f"  Dissociation Energy (ΔGdiss): {dissociation_energy} kcal/mol")
print(f"  Entropy: {entropy} kcal/mol")

if not response_data:
    print("Cannot parse data, the request was not successful or returned empty data.")

--- Assembly Summary for 1iru (Assembly 1) ---
  Components: 56 total (28-mer protein chains)
  Interfaces: 82 total
  Accessible Surface Area: 212622.4 Å²
  Buried Surface Area: 113317.18 Å²
  Dissociation Energy (ΔGdiss): 69.0 kcal/mol
  Entropy: 28.55 kcal/mol


### Cell 1.3 — Compare assemblies across entries (`1iru`, `1ryp`, `5le5`)

To place a single result in context, the same assembly-level metrics are retrieved for three
related proteasome structures and assembled into a single `pandas` DataFrame, allowing the
macromolecular size, interface count, buried surface area, and dissociation free energy to be
compared side by side across entries.

In [16]:
import pandas as pd

query_pdbids = ['1iru', '1ryp', '5le5']
query_assembly_id = '1'

assembly_results = []
for pdb_id in query_pdbids:
    response = requests.get(f'{API_point}{pdb_id}/{query_assembly_id}')
    if response.status_code == 200:
        response_data = response.json()
        assembly_data = response_data[pdb_id]['assembly']
        assembly_results.append({
            'PDB ID': pdb_id,
            'Assembly ID': query_assembly_id,
            'Macromolecular Size': f"{assembly_data.get('macromolecular_size', 'N/A')}-mer",
            'Interfaces': assembly_data.get('interface_count', 'N/A'),
            'Buried Surface Area (Å²)': assembly_data.get('buried_surface_area', 'N/A'),
            'ΔGdiss (kcal/mol)': assembly_data.get('dissociation_energy', 'N/A')
        })

df_assemblies = pd.DataFrame(assembly_results)
print(df_assemblies.to_string(index=False))

PDB ID Assembly ID Macromolecular Size  Interfaces  Buried Surface Area (Å²)  ΔGdiss (kcal/mol)
  1iru           1              28-mer          82                 113317.18              69.00
  1ryp           1              28-mer          92                 120757.64              50.28
  5le5           1              28-mer          84                 127121.21              44.56


## Section 2 — Detailed data for a single interface

The PISA *interface* endpoint returns the full characterisation of one interface within an
assembly: its area and energetics, the residues that form it, and the individual polar contacts
(hydrogen bonds and salt bridges) that stabilise it. We examine interface `25` of assembly `1`
in entry `1iru`.

### Cell 2.1 — Summarise a single interface (interface 25)

For the requested interface we report, using the paper's parameter definitions:

- **Interface area (Å²)** — the difference in total accessible surface area between the isolated
  and the interfacing structures, divided by two.
- **Solvation free-energy gain, ΔG<sup>solv</sup> (kcal/mol)** — the solvation free energy gained
  on interface formation (difference in total solvation energies of the isolated and interfacing
  structures). Negative values correspond to hydrophobic interfaces / positive protein affinity;
  this term excludes the effect of hydrogen bonds and salt bridges across the interface.
- **Stabilisation energy, ΔG<sup>stab</sup> (kcal/mol)** — the energetic contribution of the
  interface to assembly stability; more negative (favourable) values indicate a more stable
  interaction.
- **Hydrogen-bond and salt-bridge counts** — the number of specific polar contacts.
- **Complexation-significance *p*-value** — a statistical measure of interface specificity: the
  probability of obtaining a solvation energy gain lower than observed if the interface atoms were
  drawn at random from the protein surface. A value of 0.5 is unremarkable; *p* > 0.5 means the
  interface is *less* hydrophobic than expected (suggesting a crystal-packing artefact), whereas
  *p* < 0.5 indicates surprising hydrophobicity (suggesting a biologically specific interface).

For interface 25 the *p*-value of 0.572 (> 0.5) indicates an interface that is less hydrophobic
than expected by chance, consistent with a crystal-packing contact rather than a biologically
specific interaction. The identifiers of the two interacting chains are read from the interface
`molecules` list.

In [17]:
import requests, sys, json

API_point_single_interface = "https://www.ebi.ac.uk/pdbe/api/pisa/interface/"
query_pdbid = '1iru'
query_assemblyid = '1'
query_interfaceid = '25'

response_single_interface = requests.get(
    f'{API_point_single_interface}{query_pdbid}/{query_assemblyid}/{query_interfaceid}')

if response_single_interface.status_code == 200:
    interface_data = response_single_interface.json()
    print(f"--- Interface {query_interfaceid} Summary ---")
    print(f"  Interface area: {interface_data.get('interface_area')} Å²")
    print(f"  Solvation energy (ΔGsolv): {interface_data.get('solvation_energy')} kcal/mol")
    print(f"  Stabilization energy (ΔGstab): {interface_data.get('stabilization_energy')} kcal/mol")
    print(f"  Hydrogen bonds: {interface_data.get('number_hydrogen_bonds')}")
    print(f"  Salt bridges: {interface_data.get('number_salt_bridges')}")
    print(f"  p-value: {interface_data.get('p_value')}")

    molecules = interface_data.get('molecules', [])
    chains = [molecule.get('chain_id') for molecule in molecules]
    print(f"  Interacting chains: {chains}")

--- Interface 25 Summary ---
  Interface area: 766.45 Å²
  Solvation energy (ΔGsolv): -2.15 kcal/mol
  Stabilization energy (ΔGstab): -9.85 kcal/mol
  Hydrogen bonds: 14
  Salt bridges: 4
  p-value: 0.572
  Interacting chains: ['X', 'L']


### Cell 2.2 — Interface residue and contact counts

The same interface is queried and condensed to its identifier, the number of interface residues,
and the hydrogen-bond and salt-bridge counts, reported as a single descriptive sentence.

In [18]:
import requests, sys, json
import pandas as pd

API_point_single_interface = "https://www.ebi.ac.uk/pdbe/api/pisa/interface/"
query_pdbid = '1iru'
query_assemblyid = '1'
query_interfaceid = '25'

response_single_interface = requests.get(
    f'{API_point_single_interface}{query_pdbid}/{query_assemblyid}/{query_interfaceid}')
interface_data = response_single_interface.json()

interface_id = interface_data.get('interface_id')
no_interface_residues = interface_data.get('number_interface_residues')
no_hydrogen_bonds = interface_data.get('number_hydrogen_bonds')
no_salt_bridges = interface_data.get('number_salt_bridges')

print(f"The interface {interface_id} of assembly {query_pdbid} has "
      f"{no_interface_residues} interface residues with {no_hydrogen_bonds} hydrogen bonds "
      f"and {no_salt_bridges} salt bridges.")

The interface 25 of assembly 1iru has 201 interface residues with 14 hydrogen bonds and 4 salt bridges.


### Cell 2.3 — Tabulate the hydrogen-bond network

PISA encodes each contact type as a set of parallel arrays — one array per attribute, indexed
consistently across the two participating atoms. Where a `hydrogen_bonds` record is present, the
per-atom chain, residue, and atom-name arrays and the bond-distance array are read out and
recombined row by row, yielding a `df_hbonds` table in which each row is one hydrogen bond
(donor atom, acceptor atom, and the interatomic distance in Å).

In [19]:
import requests, sys, json
import pandas as pd
from IPython.display import display

API_point_single_interface = "https://www.ebi.ac.uk/pdbe/api/pisa/interface/"
query_pdbid = '1iru'
query_assemblyid = '1'
query_interfaceid = '25'

response_single_interface = requests.get(
    f'{API_point_single_interface}{query_pdbid}/{query_assemblyid}/{query_interfaceid}')
interface_data = response_single_interface.json()

if response_single_interface.status_code == 200:
    interface_pair_prop = []
    if "hydrogen_bonds" in interface_data:
        prop = interface_data["hydrogen_bonds"]
        label_seq_atom_1 = prop.get("atom_site_1_label_seq_ids", [])
        label_id_atom_1 = prop.get("atom_site_1_label_atom_ids", [])
        chain_atom_1 = prop.get("atom_site_1_chains", [])
        bond_distances = prop.get("bond_distances", [])
        label_seq_atom_2 = prop.get("atom_site_2_label_seq_ids", [])
        label_id_atom_2 = prop.get("atom_site_2_label_atom_ids", [])
        chain_atom_2 = prop.get("atom_site_2_chains", [])

        for (c1, r1, a1, c2, r2, a2, dist) in zip(
                chain_atom_1, label_seq_atom_1, label_id_atom_1,
                chain_atom_2, label_seq_atom_2, label_id_atom_2,
                bond_distances):
            interface_pair_prop.append([f"{c1}:{r1}", a1.strip(),
                                        dist,
                                        f"{c2}:{r2}", a2.strip()])

        df_hbonds = pd.DataFrame(
            interface_pair_prop, columns=[
                "Chain:ResID (Atom 1)", "Atom (Atom 1)",
                "Distance (Å)",
                "Chain:ResID (Atom 2)", "Atom (Atom 2)"])
        print(f"\nHydrogen Bonds for Interface {query_interfaceid}:")
        display(df_hbonds)
else:
    print("Cannot extract H-bonds, the request was not successful.")


Hydrogen Bonds for Interface 25:


,Chain:ResID (Atom 1),Atom (Atom 1),Distance (Å),Chain:ResID (Atom 2),Atom (Atom 2)
0,X:34,N,3.05,L:166,O
1,X:37,OG1,3.72,L:165,OH
2,X:177,NH1,2.93,L:26,O
3,X:179,N,2.73,L:24,O
4,X:203,NH1,2.94,L:191,OD1
5,X:203,NH2,3.03,L:172,O
6,X:203,NH2,3.28,L:191,OD1
7,X:32,O,2.75,L:168,N
8,X:33,OE1,3.27,L:134,OH
9,X:34,O,2.78,L:166,NH1


### Cell 2.4 — Tabulate the salt-bridge network

Salt-bridge contacts are stored with the identical parallel-array layout used for hydrogen bonds.
The same extraction and tabulation logic is therefore applied to the `salt_bridges` record,
producing a `df_salt_bridges` table directly analogous to the hydrogen-bond table and
illustrating the consistent contact schema across interaction types.

In [20]:
if "salt_bridges" in interface_data:
    sb_prop = interface_data["salt_bridges"]
    label_seq_atom_1 = sb_prop.get("atom_site_1_label_seq_ids", [])
    label_id_atom_1 = sb_prop.get("atom_site_1_label_atom_ids", [])
    chain_atom_1 = sb_prop.get("atom_site_1_chains", [])
    bond_distances = sb_prop.get("bond_distances", [])
    label_seq_atom_2 = sb_prop.get("atom_site_2_label_seq_ids", [])
    label_id_atom_2 = sb_prop.get("atom_site_2_label_atom_ids", [])
    chain_atom_2 = sb_prop.get("atom_site_2_chains", [])

    interface_pair_prop = []
    for (c1, r1, a1, c2, r2, a2, dist) in zip(
            chain_atom_1, label_seq_atom_1, label_id_atom_1,
            chain_atom_2, label_seq_atom_2, label_id_atom_2,
            bond_distances):
        interface_pair_prop.append([f"{c1}:{r1}", a1.strip(),
                                    dist,
                                    f"{c2}:{r2}", a2.strip()])

    df_salt_bridges = pd.DataFrame(
        interface_pair_prop, columns=[
            "Chain:ResID (Atom 1)", "Atom (Atom 1)",
            "Distance (Å)",
            "Chain:ResID (Atom 2)", "Atom (Atom 2)"])
    print(f"\nSalt Bridges for Interface {query_interfaceid}:")
    display(df_salt_bridges)


Salt Bridges for Interface 25:


,Chain:ResID (Atom 1),Atom (Atom 1),Distance (Å),Chain:ResID (Atom 2),Atom (Atom 2)
0,X:201,NZ,3.89,L:197,OE1
1,X:201,NZ,2.59,L:197,OE2
2,X:205,OD1,3.56,L:19,NH2
3,X:205,OD2,3.54,L:19,NH2


### Cell 2.5 — Per-residue interface profile

Each molecule of the interface is traversed residue by residue to recover the per-residue
accessible and buried surface areas (**ASA**, **BSA**), the per-residue solvation-energy
contribution (**ΔG<sub>i</sub>**), and the contact annotation. As the paper notes, the number of
residues PISA assigns to an interface includes all interface residues, whereas the table built
here retains only those that meet a filtering criterion — non-zero buried surface area, a
measurable solvation-energy contribution, or an explicit intermolecular contact. For each
retained residue, the single-letter codes read from its `residue_bonds` annotation flag the
**type of intermolecular interaction** the residue participates in — **H** = hydrogen bond,
**S** = salt bridge, **D** = disulfide bond, **C** = covalent bond — and the rows are sorted by
chain and residue number to give an ordered per-residue interface profile.

In [21]:
import requests
import pandas as pd
from IPython.display import display

API_point_single_interface = "https://www.ebi.ac.uk/pdbe/api/pisa/interface/"
query_pdbid = '1iru'
query_assemblyid = '1'
query_interfaceid = '25'

response_single_interface = requests.get(
    f'{API_point_single_interface}{query_pdbid}/{query_assemblyid}/{query_interfaceid}')

if response_single_interface.status_code == 200:
    interface_data = response_single_interface.json()
    interface_id = interface_data.get('interface_id')
    no_interface_residues = interface_data.get('number_interface_residues')
    no_hydrogen_bonds = interface_data.get('number_hydrogen_bonds')
    no_salt_bridges = interface_data.get('number_salt_bridges')

    print(f"The interface {interface_id} of assembly {query_pdbid} has "
          f"{no_interface_residues} interface residues with {no_hydrogen_bonds} hydrogen bonds "
          f"and {no_salt_bridges} salt bridges.\n")

    interface_residues_data = []
    if 'molecules' in interface_data:
        for molecule in interface_data['molecules']:
            chain_id = molecule.get('chain_id', '')
            molecule_class = molecule.get('molecule_class', '')
            residues = molecule.get('residue_label_comp_ids', [])
            seq_ids = molecule.get('residue_label_seq_ids', [])
            asa_values = molecule.get('accessible_surface_areas', [])
            bsa_values = molecule.get('buried_surface_areas', [])
            solvation_energies = molecule.get('solvation_energies', [])
            residue_bonds = molecule.get('residue_bonds', [])

            for i, (residue, seq_id) in enumerate(zip(residues, seq_ids)):
                # Only include residues that are part of the interface (BSA > 0 or involved in bonds)
                asa = asa_values[i] if i < len(asa_values) else 0
                bsa = bsa_values[i] if i < len(bsa_values) else 0
                dg = solvation_energies[i] if i < len(solvation_energies) else 0
                bond_type = residue_bonds[i] if i < len(residue_bonds) else ''

                if bsa > 0 or bond_type or abs(dg) > 0.01:
                    interface_residues_data.append({
                        'Structure': f"{chain_id}:{residue}:{seq_id}",
                        'Chain': chain_id,
                        'Residue': residue,
                        'ResID': seq_id,
                        'H': 'H' if 'H' in str(bond_type) else '',
                        'S': 'S' if 'S' in str(bond_type) else '',
                        'D': 'D' if 'D' in str(bond_type) else '',
                        'C': 'C' if 'C' in str(bond_type) else '',
                        'ASA': round(asa, 2),
                        'BSA': round(bsa, 2),
                        'ΔGi': round(dg, 2)
                    })

    df_interface_residues = pd.DataFrame(interface_residues_data)
    if not df_interface_residues.empty:
        df_interface_residues['ResID_num'] = pd.to_numeric(
            df_interface_residues['ResID'], errors='coerce')
        df_interface_residues = df_interface_residues.sort_values(
            ['Chain', 'ResID_num']).drop('ResID_num', axis=1)
        df_interface_residues.reset_index(drop=True, inplace=True)
        df_interface_residues.index += 1
        print(f"All Interface Residues for Interface {query_interfaceid}:")
        display(df_interface_residues[['Structure', 'H', 'S', 'D', 'C', 'ASA', 'BSA', 'ΔGi']])
    else:
        print("No interface residues found in the response.")
else:
    print(f"Request failed with status code: {response_single_interface.status_code}")

The interface 25 of assembly 1iru has 201 interface residues with 14 hydrogen bonds and 4 salt bridges.

All Interface Residues for Interface 25:


,Structure,H,S,D,C,ASA,BSA,ΔGi
1,L:ARG:19,H,S,,,50.20,33.36,-0.90
2,L:THR:21,,,,,48.95,1.17,0.02
3,L:ALA:24,H,,,,90.70,58.78,0.53
4,L:TYR:25,,,,,154.39,51.58,0.54
5,L:ILE:26,H,,,,60.91,55.34,0.66
6,L:ALA:27,,,,,57.27,8.34,-0.06
7,L:SER:28,,,,,38.61,7.44,0.12
8,L:GLN:29,H,,,,60.22,21.14,-0.32
9,L:SER:130,,,,,64.20,1.83,0.03
10,L:TYR:134,H,,,,89.84,29.79,-0.23


## Section 3 — All interfaces in an assembly

The PISA *interfaces* (plural) endpoint returns every interface of an assembly in a single
response. This enables assembly-wide analyses — counting interfaces, comparing their energetics,
and aggregating contacts to identify recurrent interaction sites.

### Cell 3.1 — Count the interfaces

All interfaces for assembly `1` of `1iru` are retrieved in one call, and the total
`interface_count` is read from the assembly record.

In [22]:
import requests, sys, json

API_point_all_interfaces = "https://www.ebi.ac.uk/pdbe/api/pisa/interfaces/"
query_pdbid = '1iru'
query_assemblyid = '1'

response_all_interfaces = requests.get(
    f'{API_point_all_interfaces}{query_pdbid}/{query_assemblyid}')
all_data = response_all_interfaces.json()

num_interfaces = all_data[query_pdbid]['assembly']['interface_count']
print(f"There are {num_interfaces} interfaces")

There are 82 interfaces


### Cell 3.2 — Comparative table of all interfaces

This comparative table is central to distinguishing biologically significant interfaces from
weaker crystal contacts. Two helper functions build it: `analyze_interface` extracts the area and
energetic metrics for one interface (interface area, stabilisation and solvation energies,
hydrogen-bond and salt-bridge counts, and the *p*-value), classifies the interaction from the
molecule classes of the partners (e.g. protein–protein), and records the interacting chains;
`compare_interfaces` applies it to every interface and returns a DataFrame indexed by interface
identifier. The result lets interfaces be ranked and read together — for example, an interface
combining a large area, a strongly favourable (negative) stabilisation energy, and a *p*-value
below 0.5 is a strong candidate for a specific, stable interaction.

In [23]:
import pandas as pd
from tabulate import tabulate
import requests, sys, json
from IPython.display import display


def analyze_interface(interface_data):
    analysis = {}
    analysis['interface_id'] = interface_data.get('interface_id', None)
    analysis['interface_area'] = interface_data.get('interface_area', 0.0)
    analysis['stabilization_energy'] = interface_data.get('stabilization_energy', 0.0)
    analysis['solvation_energy'] = interface_data.get('solvation_energy', 0.0)
    analysis['number_hydrogen_bonds'] = interface_data.get('number_hydrogen_bonds', 0)
    analysis['number_salt_bridges'] = interface_data.get('number_salt_bridges', 0)
    analysis['p_value'] = interface_data.get('p_value', None)

    molecules = interface_data.get('molecules', [])
    mol_classes = set(
        molecule.get('molecule_class', 'Unknown')
        for molecule in molecules
    )
    sorted_classes = sorted(list(mol_classes))
    if len(sorted_classes) == 1:
        analysis['interaction_type'] = f"{sorted_classes[0]}-{sorted_classes[0]}"
    else:
        analysis['interaction_type'] = '-'.join(sorted_classes)

    chains = sorted(list(set(
        molecule.get('chain_id')
        for molecule in molecules
        if molecule.get('chain_id') is not None
    )))
    analysis['interacting_chains'] = ', '.join(chains)
    return analysis


def compare_interfaces(assembly_data):
    interfaces = assembly_data.get('interfaces', [])
    comparison_data = []
    for interface in interfaces:
        analysis = analyze_interface(interface)
        analysis['interface_id'] = interface.get('interface_id')
        comparison_data.append(analysis)
    df = pd.DataFrame(comparison_data)
    df = df.set_index('interface_id')
    return df


if response_all_interfaces.status_code == 200:
    assembly_full_data = all_data[query_pdbid]['assembly']
    df_comparison = compare_interfaces(assembly_full_data)
    print(f"\n{len(df_comparison)} Interfaces in {query_pdbid}")
    display(df_comparison)
else:
    print("Cannot run comparison, the all-interfaces request was not successful.")


82 Interfaces in 1iru


,interface_area,stabilization_energy,solvation_energy,number_hydrogen_bonds,number_salt_bridges,p_value,interaction_type,interacting_chains
interface_id,,,,,,,,
1,1473.40,-31.99,-19.39,20,10,0.151,Protein-Protein,"I, J"
2,1472.72,-32.10,-19.50,20,10,0.158,Protein-Protein,"W, X"
3,1368.82,-17.23,-5.67,21,6,0.741,Protein-Protein,"Q, R"
4,1363.18,-17.12,-5.57,21,6,0.749,Protein-Protein,"C, D"
5,1352.64,-21.81,-12.92,15,6,0.378,Protein-Protein,"O, P"
...,...,...,...,...,...,...,...,...
78,114.70,-2.85,-2.85,0,0,0.176,Protein-Protein,"A, C"
79,97.63,-0.08,0.37,1,0,0.558,Protein-Protein,"C, G"
80,96.71,-0.07,0.37,1,0,0.559,Protein-Protein,"Q, U"


### Cell 3.3 — Interaction hotspot analysis

To locate residues that recur across the interaction network, hydrogen-bond and salt-bridge
participation is aggregated over all interfaces. Each contact contributes both partner residues,
identified by chain and UniProt sequence number; `collections.Counter` then tallies how often
each residue engages in a contact. The residues are ranked by frequency to give hydrogen-bond and
salt-bridge **hotspot** tables — positions that mediate many contacts and are therefore candidate
determinants of assembly stability.

> The cell re-derives `interfaces = assembly_full_data.get('interfaces', [])` at the top so that
> it executes independently of Cell 3.2.

In [24]:
# Initialise lists to store all interacting residues
from collections import Counter

interfaces = assembly_full_data.get('interfaces', [])

all_hbond_residues = []
all_salt_bridge_residues = []

# Loop through each interface
for interface in interfaces:
    interface_id = interface.get('interface_id')

    # Get residues from HYDROGEN BONDS
    if "hydrogen_bonds" in interface:
        prop = interface["hydrogen_bonds"]
        chains_1 = prop["atom_site_1_chains"]
        unp_nums_1 = prop["atom_site_1_unp_nums"]
        chains_2 = prop["atom_site_2_chains"]
        unp_nums_2 = prop["atom_site_2_unp_nums"]
        all_hbond_residues.extend([(f"Chain {c}", n) for c, n in zip(chains_1, unp_nums_1)])
        all_hbond_residues.extend([(f"Chain {c}", n) for c, n in zip(chains_2, unp_nums_2)])

    # Get residues from SALT BRIDGES
    if "salt_bridges" in interface:
        prop = interface["salt_bridges"]
        chains_1 = prop["atom_site_1_chains"]
        unp_nums_1 = prop["atom_site_1_unp_nums"]
        chains_2 = prop["atom_site_2_chains"]
        unp_nums_2 = prop["atom_site_2_unp_nums"]
        all_salt_bridge_residues.extend([(f"Chain {c}", n) for c, n in zip(chains_1, unp_nums_1)])
        all_salt_bridge_residues.extend([(f"Chain {c}", n) for c, n in zip(chains_2, unp_nums_2)])

# Count the frequency of each residue
hbond_counts = Counter(all_hbond_residues)
salt_bridge_counts = Counter(all_salt_bridge_residues)

df_hbond_hotspots = pd.DataFrame(hbond_counts.most_common(),
                                 columns=["Residue (Chain, UNP Num)", "H-Bond Count"])
df_salt_bridge_hotspots = pd.DataFrame(salt_bridge_counts.most_common(),
                                       columns=["Residue (Chain, UNP Num)", "Salt Bridge Count"])

print("--- Hydrogen Bond Hotspots ---")
display(df_hbond_hotspots)
print("--- Salt Bridge Hotspots ---")
display(df_salt_bridge_hotspots)

--- Hydrogen Bond Hotspots ---


,"Residue (Chain, UNP Num)",H-Bond Count
0,"(Chain J, 199)",9
1,"(Chain X, 199)",9
2,"(Chain G, 130)",6
3,"(Chain U, 130)",6
4,"(Chain I, 252)",4
...,...,...
803,"(Chain P, 3)",1
804,"(Chain C, 3)",1
805,"(Chain Q, 3)",1
806,"(Chain R, 2)",1


--- Salt Bridge Hotspots ---


,"Residue (Chain, UNP Num)",Salt Bridge Count
0,"(Chain H, 140)",6
1,"(Chain V, 140)",6
2,"(Chain I, 182)",6
3,"(Chain W, 182)",6
4,"(Chain Z, 200)",6
...,...,...
241,"(Chain M, 140)",1
242,"(Chain Q, 8)",1
243,"(Chain S, 9)",1
244,"(Chain C, 8)",1
